# Demo for RSPY-1095: Reduce the size of the UserPolicy file generated by OSAM

See: https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1095

## 1 - Initialization

In [ ]:
# Imports
import os

from resources.utils import *
from resources.widget_utils import *

from rs_common.prefect_utils import *
from rs_workflows.operation.osam_flows import osam_update_user

In [ ]:
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard_url = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard_url}")

In [ ]:
# Choose prefect deployment method
deploy_prefect_radio

In [ ]:
# Choose prefect flow run method
run_prefect_radio

## 2 - Deploy Prefect flow

In [ ]:
# Deploy the Prefect flows
s3_code_folder = f"users/{OWNER_ID}/code"
osam_update_deploy = await deploy_prefect(
    deploy_file="./osam_update_user_flow.yaml",
    s3_code_folder=s3_code_folder,
    work_pool_name=os.environ["PREFECT_WORK_POOL_MONITORING"]
)

## 3 - Run the flow

In [ ]:
# Flow params
flow_params = {
    "user_name": OWNER_ID,
    "env": {
        "owner_id": OWNER_ID,
    },
}

# Run flow
state = await run_prefect(
    deploy_name=osam_update_deploy, 
    py_func=osam_update_user, 
    params=flow_params
)

# Handle execution result
if state is not None:
    flow_run_id = state.state_details.flow_run_id
    print(f"Flow run id: {flow_run_id!r}")
    if state.is_failed() or state.is_crashed():
        # Retrieve logs of failed flow run
        response = http_session.get(f"{os.environ['PREFECT_API_URL']}/flow_runs/{flow_run_id}/logs/download")
        response.raise_for_status()
        logs_text = response.text

        print("=== Prefect flow logs ===")
        print(logs_text)
        print("=== End of logs ===")

        raise RuntimeError(f"Prefect flow {flow_run_id} failed.\n\nLogs:\n{logs_text}")

In [ ]:
# Get the UserPolicy file from the last flow run artifacts
# See: https://docs-3.prefect.io/v3/api-ref/rest-api/server/artifacts/read-latest-artifact
response = http_session.get(f"{os.environ['PREFECT_API_URL']}/artifacts/obs-rights/latest")
response.raise_for_status()
contents = response.json()["data"]

# Render the artifact as markdown
from IPython.display import Markdown
display(Markdown(contents))